In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q albumentations pyyaml
import os, re, math, random, hashlib, warnings, yaml
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "<< NO GPU, enable it in Settings >>")

In [ ]:
CONDITIONS  = ['acne','blackheads','whiteheads','dark_spots','pigmentation','pores','wrinkles','redness']
SEVERITY    = ['mild','moderate','severe']
SKIN_TYPES  = ['normal','oily','dry','combination','sensitive']

IMG_SIZE, BATCH = 256, 32
EPOCHS, FREEZE_EPOCHS, PATIENCE = 30, 2, 8
LR_BACKBONE, LR_HEADS, WD = 1e-4, 3e-4, 1e-2
W_COND, W_SEV, W_TYPE = 1.0, 0.5, 0.5
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
CKPT = Path("/kaggle/working/dermanova_best.pth")

# keywords used to map folder / class names -> condition
COND_KW = {
 'acne':['acne','pimple','papule','pustule','nodule','cyst'],
 'blackheads':['blackhead','black_head','comedone'],
 'whiteheads':['whitehead','white_head'],
 'dark_spots':['dark_spot','darkspot','age_spot','sun_spot'],
 'pigmentation':['pigment','melasma','hyperpig'],
 'pores':['pore'],
 'wrinkles':['wrinkle','fine_line'],
 'redness':['redness','rosacea','erythema'],
}

In [ ]:
IMG_EXT = {'.jpg','.jpeg','.png','.bmp','.webp'}
ROOT = Path('/kaggle/input')

def norm(s): return re.sub(r'[^a-z0-9]+','_',s.lower()).strip('_')
def match_conditions(t): t=norm(t); return [c for c,k in COND_KW.items() if any(x in t for x in k)]
def match_word(t, vocab):
    t=norm(t)
    for i,w in enumerate(vocab):
        if re.search(rf'(^|_){w}(_|$)', t): return i
    return -1
HEALTHY = re.compile(r'(^|_)(normal|healthy|clear|clean|no_\w+)(_|$)')

def ds_root(p):
    rel = p.relative_to(ROOT).parts
    return ROOT.joinpath(*rel[: 3 if rel[0]=='datasets' else 1])

def yolo_label_path(img):
    parts = list(img.parts)
    for i in range(len(parts)-1, -1, -1):
        if parts[i].lower()=='images': parts[i]='labels'; break
    return Path(*parts).with_suffix('.txt')

all_imgs = [p for p in ROOT.rglob('*') if p.suffix.lower() in IMG_EXT]
groups = defaultdict(list)
for p in all_imgs: groups[ds_root(p)].append(p)
print(f"Found {len(all_imgs)} images in {len(groups)} dataset(s)\n")

rows = []
def add_row(p, src, yc, mc, ys=-1, yt=-1):
    rows.append(dict(path=str(p), source=src, y_cond=yc, m_cond=mc, y_sev=ys, y_type=yt))

for droot, plist in groups.items():
    src = droot.name
    print("="*70); print("DATASET:", src, "| images:", len(plist))
    # ---- try YOLO format
    names = None
    for y in droot.rglob('*.y*ml'):
        try:
            d = yaml.safe_load(open(y))
            if isinstance(d, dict) and 'names' in d: names = d['names']; break
        except Exception: pass
    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
    is_yolo = names is not None and any(yolo_label_path(p).exists() for p in plist[:100])

    if is_yolo:
        print("Mode: YOLO | classes:", names)
        cmap = {i: match_conditions(n) for i,n in enumerate(names)}
        print("Class -> condition:", {names[i]: v for i,v in cmap.items()})
        covered = sorted({CONDITIONS.index(c) for v in cmap.values() for c in v})
        mc = [1.0 if j in covered else 0.0 for j in range(8)]
        for p in plist:
            lp = yolo_label_path(p); yc = [0.0]*8
            if lp.exists():
                for line in open(lp):
                    s = line.split()
                    if s:
                        for c in cmap.get(int(float(s[0])), []): yc[CONDITIONS.index(c)] = 1.0
            add_row(p, src, yc, mc)
    else:
        print("Mode: FOLDER (label = parent folder names)")
        info = []
        for p in plist:
            text = "/".join(p.relative_to(droot).parts[:-1])
            info.append((p, text, match_conditions(text), match_word(text, SEVERITY), match_word(text, SKIN_TYPES)))
        types_found = {t for *_, t in info if t >= 0}
        is_type_ds = len(types_found - {0}) >= 1       # has oily/dry/combination/sensitive folders
        dropped = defaultdict(int)
        if is_type_ds:
            print("Detected SKIN-TYPE dataset (no condition labels from this dataset)")
            for p, text, cs, sv, ty in info:
                if ty < 0: dropped[text] += 1; continue
                add_row(p, src, [0.0]*8, [0.0]*8, -1, ty)
        else:
            covered = sorted({CONDITIONS.index(c) for _,_,cs,_,_ in info for c in cs})
            mc = [1.0 if j in covered else 0.0 for j in range(8)]
            for p, text, cs, sv, ty in info:
                healthy = bool(HEALTHY.search(norm(text)))
                if not cs and not healthy: dropped[text] += 1; continue
                yc = [0.0]*8
                for c in cs: yc[CONDITIONS.index(c)] = 1.0
                add_row(p, src, yc, mc, sv if (sv >= 0 and 'acne' in cs) else -1, -1)
        if dropped: print("DROPPED (no label match):", dict(dropped))

df = pd.DataFrame(rows)
assert len(df) > 0, "No labelled images were parsed. Paste me the Cell 3 output."

# ---- remove exact duplicate files (merged datasets often contain copies)
df['md5'] = [hashlib.md5(open(p,'rb').read()).hexdigest() for p in df.path]
before = len(df); df = df.drop_duplicates('md5').reset_index(drop=True)
print(f"\nDuplicates removed: {before-len(df)} | Final images: {len(df)}")

# ---- sanity report
Y = np.stack(df.y_cond); M = np.stack(df.m_cond)
print("\nCondition | labelled-images | positives")
for j,c in enumerate(CONDITIONS): print(f"{c:13s} {int(M[:,j].sum()):7d} {int((Y[:,j]*M[:,j]).sum()):9d}")
print("Severity labels:", dict(df.y_sev[df.y_sev>=0].value_counts()))
print("Skin-type labels:", dict(df.y_type[df.y_type>=0].value_counts()))

In [ ]:
# Convert the existing labelled DataFrame into the records
# format expected by Cell A.
harmonized_dataset = df.to_dict(orient="records")

print("Records available:", len(harmonized_dataset))
print("Example keys:", harmonized_dataset[0].keys())

In [ ]:
# Cell A: clean, hash, and group (exact + near duplicates)
import hashlib
import numpy as np, cv2
from pathlib import Path

def file_md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def dhash64(img_bgr):
    g = cv2.cvtColor(np.ascontiguousarray(img_bgr), cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (9, 8), interpolation=cv2.INTER_AREA)
    return np.packbits((g[:, 1:] > g[:, :-1]).flatten()).view('>u8')[0].astype(np.uint64)

clean, hashes, flips, seen = [], [], [], set()
n_bad = n_dup = 0
for it in harmonized_dataset:
    img = cv2.imread(it['path'])
    if img is None: n_bad += 1; continue            # drop unreadable files, never fake them
    m = file_md5(it['path'])                        # FULL-file hash
    if m in seen: n_dup += 1; continue
    seen.add(m)
    clean.append(it); hashes.append(dhash64(img)); flips.append(dhash64(img[:, ::-1]))  # flip-aware
print(f"Unreadable dropped: {n_bad} | exact duplicates dropped: {n_dup} | kept: {len(clean)}")

H, HF = np.array(hashes, dtype=np.uint64), np.array(flips, dtype=np.uint64)
POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def popcnt(x): return POP[x.view(np.uint8).reshape(*x.shape, 8)].sum(-1)

n = len(clean)
parent = list(range(n))
def find(a):
    while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

# (1) Roboflow augmentation copies share a filename prefix
first = {}
for i, it in enumerate(clean):
    fname = Path(it['path']).name
    if '.rf.' not in fname: continue
    k = fname.split('.rf.')[0]
    if k in first: union(i, first[k])
    else: first[k] = i

# (2) near-duplicates by image content (works ACROSS both datasets)
THR = 5
for s in range(0, n, 256):
    blk = H[s:s+256]
    d = np.minimum(popcnt(blk[:, None] ^ H[None, :]), popcnt(blk[:, None] ^ HF[None, :]))
    ii, jj = np.where(d <= THR)
    for a, b in zip(ii + s, jj):
        if a < b: union(int(a), int(b))

groups = np.array([find(i) for i in range(n)])
sizes = np.bincount(np.unique(groups, return_inverse=True)[1])
print(f"Groups: {len(sizes)} | largest group: {sizes.max()} | groups with >1 image: {(sizes>1).sum()}")

In [ ]:

# Cell B: grouped train/validation/test split + leakage audit

from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

# Use existing multi-label condition targets
Y = np.stack([it['y_cond'] for it in clean]).astype(np.float32)

# Ensure groups is aligned with clean
assert len(Y) == len(groups) == len(clean), (
    f"Length mismatch: Y={len(Y)}, groups={len(groups)}, clean={len(clean)}"
)

# Stratify approximately by the rarest positive condition
rarity = Y / np.maximum(Y.sum(axis=0), 1)
strat = np.where(Y.sum(axis=1) > 0, rarity.argmax(axis=1), -1)

idx = np.arange(len(clean))

# First split: approximately 5/6 train+validation, 1/6 test
outer = StratifiedGroupKFold(
    n_splits=6, shuffle=True, random_state=42
)
trval_rel, test_idx = next(
    outer.split(idx, strat, groups)
)
trval_idx = idx[trval_rel]

# Second split: divide train+validation into approximately 70/15 overall
inner = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=7
)
train_rel, val_rel = next(
    inner.split(trval_idx, strat[trval_idx], groups[trval_idx])
)

train_idx = trval_idx[train_rel]
val_idx = trval_idx[val_rel]

# Confirm no group crosses splits
train_groups = set(groups[train_idx])
val_groups = set(groups[val_idx])
test_groups = set(groups[test_idx])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

def split_report(name, ix):
    print(
        f"{name:5s} {len(ix):6d} imgs | "
        f"positives per condition: {Y[ix].sum(0).astype(int).tolist()}"
    )

for name, ix in [
    ("train", train_idx),
    ("val", val_idx),
    ("test", test_idx),
]:
    split_report(name, ix)

print("\nActual split percentages:")
for name, ix in [
    ("train", train_idx),
    ("val", val_idx),
    ("test", test_idx),
]:
    print(f"{name}: {len(ix) / len(clean) * 100:.2f}%")

# Audit for remaining look-alikes across splits
def residual_near_dups(a_idx, b_idx, thr=10):
    if len(a_idx) == 0 or len(b_idx) == 0:
        return 0

    cnt = 0
    for s in range(0, len(a_idx), 256):
        blk = H[a_idx[s:s+256]]
        d = np.minimum(
            popcnt(blk[:, None] ^ H[b_idx][None, :]),
            popcnt(blk[:, None] ^ HF[b_idx][None, :])
        )
        cnt += int((d.min(axis=1) <= thr).sum())
    return cnt

print(
    "\nVal images with a look-alike in train (dist<=10):",
    residual_near_dups(val_idx, train_idx)
)
print(
    "Test images with a look-alike in train (dist<=10):",
    residual_near_dups(test_idx, train_idx)
)


In [ ]:

# Cell C1: Define the image preprocessing enhancer

class SkinImageEnhancer:
    def __init__(self, img_size=256):
        self.img_size = img_size

    def preprocess_image(self, path):
        img = cv2.imread(str(path))

        if img is None:
            raise ValueError(f"Cannot read image: {path}")

        # OpenCV loads BGR; convert to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Resize to the model's expected input dimensions
        img = cv2.resize(
            img,
            (self.img_size, self.img_size),
            interpolation=cv2.INTER_AREA
        )

        return img

enhancer = SkinImageEnhancer(IMG_SIZE)

print("Enhancer initialized.")
print("Image size:", IMG_SIZE, "x", IMG_SIZE)


In [ ]:
# Cell C: honest Dataset + loaders
import random, torch
from torch.utils.data import Dataset, DataLoader

class HarmonizedSkinDataset(Dataset):
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    def __init__(self, records, enhancer, is_train=False):
        self.records, self.enhancer, self.is_train = records, enhancer, is_train
    def __len__(self): return len(self.records)            # no padding, no fake samples
    def __getitem__(self, i):
        it = self.records[i]
        img = self.enhancer.preprocess_image(it['path'])   # fail loudly, files were pre-checked
        if self.is_train and random.random() < 0.5: img = img[:, ::-1]
        x = torch.from_numpy(np.ascontiguousarray(img)).float().permute(2, 0, 1) / 255.0
        x = (x - self.MEAN) / self.STD
        y = torch.tensor(it['target'], dtype=torch.float32)
        mask = torch.tensor(it.get('mask', np.ones(8)), dtype=torch.float32)  # 0 = condition not labelled in this source
        return x, y, mask

mk = lambda ix, tr: HarmonizedSkinDataset([clean[i] for i in ix], enhancer, is_train=tr)
train_loader = DataLoader(mk(train_idx, True),  batch_size=32, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(mk(val_idx, False),   batch_size=32, num_workers=2)
test_loader  = DataLoader(mk(test_idx, False),  batch_size=32, num_workers=2)

In [ ]:

print("Available keys:", clean[0].keys())
print("Condition labels:", clean[0]['y_cond'])
print("Condition mask:", clean[0]['m_cond'])


In [ ]:

class HarmonizedSkinDataset(Dataset):
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __init__(self, records, enhancer, is_train=False):
        self.records = records
        self.enhancer = enhancer
        self.is_train = is_train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        it = self.records[i]
        img = self.enhancer.preprocess_image(it['path'])

        if self.is_train and random.random() < 0.5:
            img = img[:, ::-1, :]

        x = torch.from_numpy(
            np.ascontiguousarray(img)
        ).float().permute(2, 0, 1) / 255.0

        x = (x - self.MEAN) / self.STD
        y = torch.tensor(it['y_cond'], dtype=torch.float32)
        mask = torch.tensor(it['m_cond'], dtype=torch.float32)

        return x, y, mask


def mk(ix, tr):
    return HarmonizedSkinDataset(
        [clean[int(i)] for i in ix],
        enhancer,
        is_train=tr
    )

train_loader = DataLoader(
    mk(train_idx, True), batch_size=32,
    shuffle=True, drop_last=False, num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    mk(val_idx, False), batch_size=32,
    shuffle=False, num_workers=0
)

test_loader = DataLoader(
    mk(test_idx, False), batch_size=32,
    shuffle=False, num_workers=0
)

print("Dataset and loaders rebuilt.")


In [ ]:

images, labels, masks = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Masks:", masks.shape)

assert labels.shape[1] == 8
assert masks.shape == labels.shape
print("Batch test passed.")


In [ ]:
class DermaNova(nn.Module):
    def __init__(self, n_cond=8, n_type=5, pretrained=True):
        super().__init__()
        b = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        self.features, self.pool = b.features, nn.AdaptiveAvgPool2d(1)
        self.trunk = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280,512), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(0.2))
        self.head_cond = nn.Linear(512, n_cond)
        self.head_sev  = nn.Linear(512, 2)        # ordinal: P(sev>mild), P(sev>moderate)
        self.head_type = nn.Linear(512, n_type)
    def forward(self, x):
        z = self.trunk(self.pool(self.features(x)).flatten(1))
        return self.head_cond(z), self.head_sev(z), self.head_type(z)

model = DermaNova().to(device)
# set the rare-positive prior BEFORE training (not after)
prior = 0.15
with torch.no_grad(): model.head_cond.bias.fill_(-math.log((1-prior)/prior))

In [ ]:
def asl_masked(logits, targets, mask, g_neg=4.0, g_pos=1.0, clip=0.05, eps=1e-8):
    p = torch.sigmoid(logits)
    pos = targets * torch.log(p.clamp(min=eps)) * (1-p)**g_pos
    ps = (p - clip).clamp(min=0)
    neg = (1-targets) * torch.log((1-ps).clamp(min=eps)) * ps**g_neg
    return -((pos+neg)*mask).sum() / logits.size(0)

def compute_loss(lc, ls, lt, b):
    yc, mc = b['y_cond'].to(device), b['m_cond'].to(device)
    ys, yt = b['y_sev'].to(device), b['y_type'].to(device)
    lc, ls, lt = lc.float(), ls.float(), lt.float()
    zero = lc.sum()*0
    l_c = asl_masked(lc, yc, mc)
    v = ys >= 0
    if v.any():
        t = torch.stack([(ys[v]>0), (ys[v]>1)], 1).float()
        l_s = F.binary_cross_entropy_with_logits(ls[v], t)
    else: l_s = zero
    v = yt >= 0
    l_t = F.cross_entropy(lt[v], yt[v]) if v.any() else zero
    return W_COND*l_c + W_SEV*l_s + W_TYPE*l_t, (l_c.item(), l_s.item(), l_t.item())

In [ ]:
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
ac = lambda: torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp)

def set_backbone_trainable(flag):
    for p in model.features.parameters(): p.requires_grad = flag

def train_epoch(dl, opt, frozen):
    model.train()
    if frozen: model.features.eval()      # keep BN stats of frozen backbone untouched
    tot = n = 0
    for b in dl:
        x = b['img'].to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with ac(): lc, ls, lt = model(x)
        loss, _ = compute_loss(lc, ls, lt, b)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(opt); scaler.update()
        tot += loss.item(); n += 1
    return tot / max(n,1)

@torch.no_grad()
def predict(dl):
    model.eval(); o = defaultdict(list); tot = n = 0
    for b in dl:
        with ac(): lc, ls, lt = model(b['img'].to(device, non_blocking=True))
        loss, _ = compute_loss(lc, ls, lt, b); tot += loss.item(); n += 1
        o['pc'].append(torch.sigmoid(lc.float()).cpu()); o['ps'].append(torch.sigmoid(ls.float()).cpu())
        o['pt'].append(F.softmax(lt.float(),1).cpu())
        for k in ('y_cond','m_cond','y_sev','y_type'): o[k].append(b[k])
    P = {k: torch.cat(v).numpy() for k,v in o.items()}; P['loss'] = tot/max(n,1); return P

def metrics(P):
    aucs = {}
    for j,c in enumerate(CONDITIONS):
        v = P['m_cond'][:,j] > 0; y = P['y_cond'][v,j]
        if v.sum() > 0 and 0 < y.sum() < v.sum(): aucs[c] = roc_auc_score(y, P['pc'][v,j])
    v = P['y_sev'] >= 0
    sev_acc = float((((P['ps'][v] > 0.5).sum(1)) == P['y_sev'][v]).mean()) if v.any() else np.nan
    v = P['y_type'] >= 0
    type_f1 = f1_score(P['y_type'][v], P['pt'][v].argmax(1), average='macro') if v.any() else np.nan
    mean_auc = float(np.mean(list(aucs.values()))) if aucs else np.nan
    score = float(np.nanmean([mean_auc, sev_acc, type_f1]))
    return dict(aucs=aucs, mean_auc=mean_auc, sev_acc=sev_acc, type_f1=type_f1, score=score)

In [ ]:

# Re-create the SkinDS dataset class

class SkinDS(Dataset):
    def __init__(self, d, tf):
        self.d = d.reset_index(drop=True)
        self.tf = tf

    def __len__(self):
        return len(self.d)

    def __getitem__(self, i):
        r = self.d.iloc[i]

        img = cv2.imread(str(r["path"]))
        if img is None:
            raise RuntimeError(f"Unreadable image: {r['path']}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.tf(image=img)["image"]

        return {
            "img": img,
            "y_cond": torch.tensor(
                r["y_cond"], dtype=torch.float32
            ),
            "m_cond": torch.tensor(
                r["m_cond"], dtype=torch.float32
            ),
            "y_sev": torch.tensor(
                int(r["y_sev"]), dtype=torch.long
            ),
            "y_type": torch.tensor(
                int(r["y_type"]), dtype=torch.long
            ),
        }

print("SkinDS defined successfully.")


In [ ]:

names = [
    "df", "clean", "groups", "Y",
    "train_idx", "val_idx", "test_idx",
    "train_df", "val_df", "test_df",
    "train_tf", "val_tf", "SkinDS"
]

for name in names:
    print(f"{name}: {'EXISTS' if name in globals() else 'MISSING'}")


In [ ]:

import pandas as pd
import numpy as np

clean_df = pd.DataFrame(clean).reset_index(drop=True)

# Verify the split indices belong to the cleaned dataset
assert len(clean_df) == len(groups) == len(Y), (
    "Dataset, duplicate groups, and labels have different lengths."
)

# Check that indices are valid and splits don't overlap
train_idx = np.asarray(train_idx, dtype=int)
val_idx = np.asarray(val_idx, dtype=int)
test_idx = np.asarray(test_idx, dtype=int)

assert all(
    len(idx) > 0 and idx.min() >= 0 and idx.max() < len(clean_df)
    for idx in [train_idx, val_idx, test_idx]
), "A split is empty or contains an invalid index."

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

# Verify duplicate groups do not cross splits
assert set(groups[train_idx]).isdisjoint(set(groups[val_idx]))
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))
assert set(groups[val_idx]).isdisjoint(set(groups[test_idx]))

train_df = clean_df.iloc[train_idx].reset_index(drop=True)
val_df = clean_df.iloc[val_idx].reset_index(drop=True)
test_df = clean_df.iloc[test_idx].reset_index(drop=True)

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("Duplicate-group leakage check passed.")


In [ ]:

import albumentations as A
from albumentations.pytorch import ToTensorV2

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Affine(
        scale=(0.9, 1.1),
        rotate=(-15, 15),
        translate_percent=(-0.05, 0.05),
        p=0.7
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.5
    ),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

print("Training and validation transformations restored.")


In [ ]:

kw = {
    "batch_size": BATCH,
    "num_workers": 2,
    "pin_memory": device.type == "cuda"
}

train_dl = DataLoader(
    SkinDS(train_df, train_tf),
    shuffle=True,
    drop_last=(len(train_df) >= BATCH),
    **kw
)

val_dl = DataLoader(
    SkinDS(val_df, val_tf),
    shuffle=False,
    **kw
)

test_dl = DataLoader(
    SkinDS(test_df, val_tf),
    shuffle=False,
    **kw
)

print("Train batches:", len(train_dl))
print("Validation batches:", len(val_dl))
print("Test batches:", len(test_dl))


In [ ]:

b = next(iter(train_dl))

print("Images:", b["img"].shape)
print("Condition labels:", b["y_cond"].shape)
print("Condition masks:", b["m_cond"].shape)
print("Severity labels:", b["y_sev"].shape)
print("Skin-type labels:", b["y_type"].shape)

assert b["img"].shape[1:] == (3, IMG_SIZE, IMG_SIZE)
assert b["y_cond"].shape[1] == 8
assert b["m_cond"].shape[1] == 8

print("Training batch passed.")


In [ ]:
head_params = [p for n,p in model.named_parameters() if not n.startswith('features.')]
opt = torch.optim.AdamW([{'params': model.features.parameters(), 'lr': LR_BACKBONE},
                         {'params': head_params, 'lr': LR_HEADS}], weight_decay=WD)
sched, best, bad = None, -1, 0
set_backbone_trainable(False)

for ep in range(EPOCHS):
    if ep == FREEZE_EPOCHS:
        set_backbone_trainable(True)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS-FREEZE_EPOCHS, eta_min=1e-6)
    tr = train_epoch(train_dl, opt, frozen=(ep < FREEZE_EPOCHS))
    P = predict(val_dl); m = metrics(P)
    if sched: sched.step()
    print(f"ep {ep+1:02d} | train {tr:.4f} | val {P['loss']:.4f} | meanAUC {m['mean_auc']:.3f} "
          f"| sevAcc {m['sev_acc']:.3f} | typeF1 {m['type_f1']:.3f} | score {m['score']:.3f}")
    if m['score'] > best:
        best, bad = m['score'], 0
        torch.save(model.state_dict(), CKPT)
    else:
        bad += 1
        if bad >= PATIENCE: print("Early stopping."); break
print("Best val score:", round(best,4))

In [ ]:
model.load_state_dict(torch.load(CKPT)); 
Pv, Pt = predict(val_dl), predict(test_dl)

thresholds = []
for j,c in enumerate(CONDITIONS):
    v = Pv['m_cond'][:,j] > 0; y = Pv['y_cond'][v,j]
    if v.sum() == 0 or y.sum() == 0: thresholds.append(0.5); continue
    cands = np.arange(0.05, 0.96, 0.05)
    f1s = [f1_score(y, Pv['pc'][v,j] > t, zero_division=0) for t in cands]
    thresholds.append(float(cands[int(np.argmax(f1s))]))

m = metrics(Pt)
print("TEST RESULTS (thresholds tuned on val)\n")
print(f"{'condition':13s} {'AUC':>6s} {'F1':>6s} {'thr':>5s}")
for j,c in enumerate(CONDITIONS):
    v = Pt['m_cond'][:,j] > 0
    if c in m['aucs']:
        f1 = f1_score(Pt['y_cond'][v,j], Pt['pc'][v,j] > thresholds[j], zero_division=0)
        print(f"{c:13s} {m['aucs'][c]:6.3f} {f1:6.3f} {thresholds[j]:5.2f}")
    else: print(f"{c:13s}   n/a (no labels or only one class in test)")
print("\nSeverity acc:", m['sev_acc'], "| Skin-type macro-F1:", m['type_f1'])
heads_trained = dict(conditions=True, severity=bool((df.y_sev>=0).any()), skin_type=bool((df.y_type>=0).any()))
print("Heads with real training data:", heads_trained)

In [ ]:

for split_name, split_df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:
    print(f"\n--- {split_name} ---")

    for col in ["y_sev", "y_type"]:
        labels = split_df[col].astype(int)
        valid = labels[labels >= 0]

        print(
            f"{col}: valid={len(valid)}/{len(labels)}, "
            f"distribution={valid.value_counts().sort_index().to_dict()}"
        )

print("\nLabel encodings expected by the current code:")
print("Severity: 0=mild, 1=moderate, 2=severe")
print("Skin type: 0–4, matching your defined skin-type class order")


In [ ]:
FINAL = Path("/kaggle/working/dermanova_final.pth")
torch.save(dict(model_state_dict=model.state_dict(), backbone='efficientnet_b0',
                conditions=CONDITIONS, severity_classes=SEVERITY, skin_types=SKIN_TYPES,
                thresholds=dict(zip(CONDITIONS, thresholds)), img_size=IMG_SIZE, mean=MEAN, std=STD,
                heads_trained=heads_trained, severity_mode='ordinal_cumulative_2logits',
                test_metrics=dict(mean_auc=m['mean_auc'], per_class_auc=m['aucs'])), FINAL)
print("Saved", FINAL, f"{FINAL.stat().st_size/1e6:.1f} MB")

In [ ]:
def gradcam(x, idx):
    model.eval()
    with torch.enable_grad():
        f = model.features(x)
        logit = model.head_cond(model.trunk(model.pool(f).flatten(1)))[0, idx]
        g = torch.autograd.grad(logit, f)[0]
    cam = F.relu((g.mean((2,3), keepdim=True) * f).sum(1)).squeeze(0).detach().cpu().numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE)); return (cam - cam.min()) / (cam.max() + 1e-8)

ds = SkinDS(val_df, val_tf)
cidx = 0  # 0 = acne; change to any condition index
pos = [i for i in range(len(ds)) if ds.d.iloc[i].y_cond[cidx] == 1 and ds.d.iloc[i].m_cond[cidx] == 1][:4]
fig, ax = plt.subplots(2, max(len(pos),1), figsize=(4*max(len(pos),1), 8), squeeze=False)
for k, i in enumerate(pos):
    x = ds[i]['img'].unsqueeze(0).to(device)
    img = (x[0].cpu().permute(1,2,0).numpy() * np.array(STD) + np.array(MEAN)).clip(0,1)
    cam = gradcam(x, cidx)
    heat = cv2.applyColorMap((cam*255).astype(np.uint8), cv2.COLORMAP_TURBO)[:,:,::-1] / 255
    ax[0,k].imshow(img); ax[0,k].axis('off')
    ax[1,k].imshow(0.55*img + 0.45*heat); ax[1,k].axis('off')
plt.suptitle(f"Grad-CAM: {CONDITIONS[cidx]}"); plt.show()

In [ ]:
from IPython.display import FileLink
FileLink("dermanova_final.pth")